In [3]:
import pandas as pd
import numpy as np

# Load raw orders data
orders = pd.read_csv("/content/drive/MyDrive/capstone project/orders.csv")

# 1. Standardize and parse date columns
orders['order_date'] = pd.to_datetime(orders['order_date'])

# 🚨 CRITICAL: Define your snapshot date from the Data Dictionary here
# (Example placeholder: '2026-01-01'. Replace with your actual project snapshot date)
SNAPSHOT_DATE = pd.to_datetime('2026-01-01')

# 2. Prevent Data Leakage: Filter out future transactional records
valid_orders = orders[orders['order_date'] <= SNAPSHOT_DATE].copy()

print(f"Total historical orders: {len(orders)}")
print(f"Valid orders on or before snapshot ({SNAPSHOT_DATE.date()}): {len(valid_orders)}")

# 3. Aggregate to create RFM Features
# Recency: Days between customer's latest valid purchase and the snapshot date
# Frequency: Count of unique valid orders placed
# Monetary: Sum of total spend across valid orders
rfm = valid_orders.groupby('customer_id').agg(
    Latest_Purchase=('order_date', 'max'),
    Frequency=('order_id', 'nunique'),
    Monetary=('gross_amount', 'sum') # Changed 'total_amount' to 'gross_amount'
).reset_index()

rfm['Recency'] = (SNAPSHOT_DATE - rfm['Latest_Purchase']).dt.days

# Drop intermediate timestamp column
rfm = rfm.drop(columns=['Latest_Purchase'])

print("\n=== Initial RFM Base Table ===")
print(rfm.head())

Total historical orders: 10009
Valid orders on or before snapshot (2026-01-01): 10009

=== Initial RFM Base Table ===
  customer_id  Frequency  Monetary  Recency
0   CUST00001          6   2955.57      200
1   CUST00002          3   1713.10       67
2   CUST00003          1    649.98      264
3   CUST00004          1   1604.04      224
4   CUST00005          6   3910.43       43


In [4]:
# Create quintile scores (1 to 5) where 5 is the optimal business behavior
# For Recency: a lower number of days since last purchase is better -> labels=[5, 4, 3, 2, 1]
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop')
# For Frequency & Monetary: higher is better -> labels=[1, 2, 3, 4, 5]
rfm['F_Score'] = pd.qcut(rfm['Frequency'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')

# Convert categories to integers for conditional logic operations
for col in ['R_Score', 'F_Score', 'M_Score']:
    rfm[col] = rfm[col].astype(int)

# Define segment designation logic based on behavior scores
def assign_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']

    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions' # Recently, frequently, and spend highly
    elif r <= 2 and f >= 3:
        return 'At-Risk Customers' # Used to buy frequently, but haven't returned recently
    elif r <= 2 and f <= 2:
        return 'Dormant Customers' # Low engagement, long gone
    elif r >= 4 and f <= 2:
        return 'New Customers' # Recent buyers, but low historical frequency
    elif f >= 4 and m <= 2:
        return 'Discount-Sensitive' # Buy often but spend very little per order
    else:
        return 'Regular Loyalists' # Mid-tier baseline customers

rfm['segment_name'] = rfm.apply(assign_segment, axis=1)

print("\n=== Segment Distribution Count ===")
print(rfm['segment_name'].value_counts())


=== Segment Distribution Count ===
segment_name
Regular Loyalists     701
Dormant Customers     620
Champions             433
At-Risk Customers     339
New Customers         304
Discount-Sensitive      3
Name: count, dtype: int64


In [6]:
# Load support ticket dataset
tickets = pd.read_csv("/content/drive/MyDrive/capstone project/support_tickets.csv")

# Filter support tickets up to snapshot date to prevent leakage
valid_tickets = tickets[pd.to_datetime(tickets['ticket_date']) <= SNAPSHOT_DATE]

# Calculate total support complaints per customer
ticket_summary = valid_tickets.groupby('customer_id').size().reset_index(name='support_complaints')

# Merge non-RFM behavioral metrics into the primary customer matrix
final_segments_df = pd.merge(rfm, ticket_summary, on='customer_id', how='left').fillna(0)
final_segments_df['support_complaints'] = final_segments_df['support_complaints'].astype(int)

# Export the required structured artifact
final_output_cols = ['customer_id', 'segment_name', 'Recency', 'Frequency', 'Monetary', 'support_complaints']
final_segments_df[final_output_cols].to_csv("segments.csv", index=False)
print("\n✅ Successfully exported segments.csv with integrated behavioral signals!")


✅ Successfully exported segments.csv with integrated behavioral signals!
